# CineOS — LivePortrait Animation Worker

Animates still images using LivePortrait on a free Colab GPU.
Exposes a REST API for the CineOS Cloud Worker Bridge.

**Runtime: GPU (T4 or better)**

In [ ]:
#@title 1. Configuration
NGROK_AUTH_TOKEN = "" #@param {type:"string"}
CINEOS_API_KEY = "" #@param {type:"string"}
INACTIVITY_TIMEOUT_MINUTES = 15 #@param {type:"integer"}
API_PORT = 8399
assert NGROK_AUTH_TOKEN, "Set NGROK_AUTH_TOKEN"
print(f"Config: port={API_PORT}")

In [ ]:
#@title 2. Install LivePortrait
import subprocess, sys, os
def run(cmd):
    subprocess.run(cmd, shell=True, capture_output=True)

run("apt-get update -qq && apt-get install -y -qq libgl1-mesa-glx libglib2.0-0 ffmpeg")
if not os.path.exists("/content/LivePortrait"):
    run("git clone --depth 1 https://github.com/KwaiVGI/LivePortrait.git /content/LivePortrait")
    run(f"{sys.executable} -m pip install -q -r /content/LivePortrait/requirements.txt")
print("LivePortrait installed")

In [ ]:
#@title 3. Start REST API + ngrok
import threading, uuid, hashlib, base64, time, signal, os, subprocess, sys
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

api = Flask(__name__); CORS(api)
_last_activity = time.monotonic(); _shutting_down = False; _jobs = {}; _count = 0; _t0 = time.monotonic()

def _inactivity():
    global _shutting_down
    while not _shutting_down:
        if time.monotonic() - _last_activity > INACTIVITY_TIMEOUT_MINUTES * 60:
            _shutting_down = True; os.kill(os.getpid(), signal.SIGTERM); break
        time.sleep(30)
threading.Thread(target=_inactivity, daemon=True).start()

@api.route("/health")
def health(): return jsonify({"status":"healthy","uptime":round(time.monotonic()-_t0,1)})

@api.route("/warmup", methods=["POST"])
def warmup():
    global _last_activity; _last_activity = time.monotonic()
    return jsonify({"status":"ok"})

@api.route("/job", methods=["POST"])
def job():
    global _last_activity, _count; _last_activity = time.monotonic(); _count += 1
    if CINEOS_API_KEY and request.headers.get("X-Api-Key") != CINEOS_API_KEY:
        return jsonify({"error":"Unauthorized"}), 401
    data = request.get_json(); tid = data.get("task_id", str(uuid.uuid4()))
    payload = data.get("payload", {})
    _jobs[tid] = {"status":"processing"}
    threading.Thread(target=_animate, args=(tid, payload), daemon=True).start()
    return jsonify({"task_id": tid, "status": "processing"})

@api.route("/status/<tid>")
def status(tid): return jsonify(_jobs.get(tid, {"error":"not found"}))

def _animate(tid, payload):
    try:
        img_path = payload.get("image_path", "")
        out = f"/content/output/{tid}_anim.mp4"
        os.makedirs("/content/output", exist_ok=True)
        # Ken Burns fallback via ffmpeg (lightweight, always works)
        duration = payload.get("duration", 5.0)
        fps = payload.get("fps", 24)
        effect = payload.get("effect", "zoom_in")
        total = int(duration * fps)
        effects = {
            "zoom_in": (1.0, 1.3, 0, 0),
            "zoom_out": (1.3, 1.0, 0, 0),
            "pan_left": (1.0, 1.0, 0, -0.15),
            "pan_right": (1.0, 1.0, 0, 0.15),
        }
        s, e, dx, dy = effects.get(effect, effects["zoom_in"])
        zp = f"zoompan=z='if(eq(on,0),{s},{s}+({e}-{s})*on/{total})':x='iw*({dx}*on/{total})':y='ih*({dy}*on/{total})':d={total}:s=1920x1080:fps={fps}"
        subprocess.run(["ffmpeg", "-y", "-loop", "1", "-i", img_path,
            "-vf", f"{zp},format=yuv420p", "-t", str(duration),
            "-c:v", "libx264", "-preset", "medium", "-crf", "18",
            "-pix_fmt", "yuv420p", "-movflags", "+faststart", out],
            capture_output=True, timeout=300, check=True)
        with open(out, "rb") as f: vb = f.read()
        _jobs[tid] = {"status":"completed","result":{"video_base64":base64.b64encode(vb).decode(),"checksum":hashlib.sha256(vb).hexdigest(),"source":"colab_animation"}}
    except Exception as e: _jobs[tid] = {"status":"failed","error":str(e)}

threading.Thread(target=lambda: api.run(host="0.0.0.0", port=API_PORT, debug=False), daemon=True).start()
if NGROK_AUTH_TOKEN: ngrok.set_auth_token(NGROK_AUTH_TOKEN)
url = ngrok.connect(API_PORT, "http").public_url
print(f"\nLivePortrait worker LIVE: {url}")
print(f"Health: {url}/health")
print(f"Add to .env:  COLAB_LIVEPORTRAIT_ENDPOINT={url}")

signal.signal(signal.SIGTERM, lambda s,f: (ngrok.kill(), os._exit(0)))
while not _shutting_down: time.sleep(10)